# Police Station Spatial Intelligence — Centrality

## //00 Setup | Import Libraries
> All topologicpy modules needed for geometry, topology, and graph analysis.

In [ ]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## //01 Version Check
> Confirm topologicpy meets the minimum required version (0.9.31+).

In [ ]:
print("This tutorial requires topologicpy version 0.9.31 or newer.")
print(Helper.Version())

## //02 Renderer | Configuration
> Set render target. Options: `vscode` | `colab` | `browser`.

In [ ]:
renderer = "vscode"

## //03 Utility Functions
> `reset_dictionaries` — clear face metadata | `transfer_dicts_by_key` — propagate graph values back to geometry.

In [ ]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts[str(value)]
            f = Topology.SetDictionary(f, d)


## //04 Import Floor Plan
> Load the processed ground-floor BREP face from `gf-floor-plan-face.brep`.

In [ ]:
police_station = Topology.ByBREPPath(r"C:\Users\chidi\macad\module-03\aia-gml\phase-02-spatial-analysis\assets\gf-floor-plan-face.brep")

## //05 Visualize Geometry
> Raw floor plan — single face, no grid.

In [ ]:
Topology.Show(police_station,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## //06 Grid Overlay
> Compute bounding rectangle and generate a 2×2 unit edge grid clipped to the floor plan.

In [ ]:
b_r = Wire.BoundingRectangle(police_station)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
uRange = list(range(0,int(width)+2,2))
vRange = list(range(0,int(length)+2,2))

grid = Grid.EdgesByDistances(police_station, clip=True, uRange=uRange, vRange=vRange)

## //07 Visualize Geometry | Grid

In [ ]:
Topology.Show(police_station, grid,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="grey",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## //08 Slice Floor Plan | Shell
> Divide the face into a regular grid of cells. Each cell becomes a graph node.

In [ ]:
shell = Topology.Slice(police_station, grid)
faces = Topology.Faces(shell)
# Assign a sequential unique face id to reference it later (e.g. "face_21")
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)

## //09 Visualize Shell

In [ ]:
Topology.Show(shell,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=0.9,
              edgeColor="black",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## //10 Derive Graphs | Navigation & Analysis
> Navigation graph: `direct=False, viaSharedTopologies=True` | Analysis graph: direct adjacency.

In [ ]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they 
navigation_graph = Graph.ByTopology(shell, direct=False, viaSharedTopologies=True)
analysis_graph = Graph.ByTopology(shell)

## //11 Store Graph Vertices

In [ ]:
g_verts = Graph.Vertices(analysis_graph)

## //12 Visualize Analysis Graph

In [ ]:
Topology.Show(analysis_graph,
              camera=[0,0,6],
              vertexSize=4,
              vertexColor="red",
              edgeColor="lightgrey",
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)
              

## //13 Spatial Analysis

### //13a Minimum Spanning Tree
> MST demonstrated on a simple 3-prism graph — too computationally expensive on the full grid.

In [ ]:
cc1 = CellComplex.Prism()
cc2 = Topology.Translate(cc1, 1.1, 0, 0)
cc3 = Topology.Translate(cc2, 1.1, 0, 0)
g1 = Graph.ByTopology(cc1)
g2 = Graph.ByTopology(cc2)
g3 = Graph.ByTopology(cc3)
g2 = Graph.MinimumSpanningTree(g2)
g3 = Graph.Complete(g3)
Topology.Show(g1, g2, g3,
              vertexSize=12,
              vertexColor="red",
              edgeColor="lightgrey",
              edgeWidth=4,
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)


In [ ]:
dn1 = Graph.Density(g1)
dn2 = Graph.Density(g2)
dn3 = Graph.Density(g3)
print("Density 1:", dn1)
print("Density 2:", dn2)
print("Density 3:", dn3)

In [ ]:
dr1 = Graph.Diameter(g1)
dr2 = Graph.Diameter(g2)
dr3 = Graph.Diameter(g3)
print("Diameter 1:", dr1)
print("Diameter 2:", dr2)
print("Diameter 3:", dr3)

### //13b Shortest Path | Navigation Graph

In [ ]:
import time

start_vertex = Vertex.ByCoordinates(xmin+2, ymax-2,0) # Upper left corner
end_vertex = Vertex.ByCoordinates(xmax-2,ymin+2,0) # Lower right corner
crg = Graph.CompiledRoutingGraph(navigation_graph, precomputeTurns=False)
start = time.time()
shortest_path = Graph.ShortestPath(crg, vertexA=start_vertex, vertexB=end_vertex)
end = time.time()
print("Shortest Path Duration:", round(end-start, 2), "seconds")

# Straighten the shortest path (optional)
start = time.time()
straight_path = Wire.Straighten(shortest_path, host=police_station)
end = time.time()
print("Straighten Wire Duration:", round(end-start, 2), "seconds")

print("Original Shortest Path Length:", round(Wire.Length(shortest_path), 2))
print("Straightened Shortened Path Length:", round(Wire.Length(straight_path), 2))
edges = Topology.Edges(shortest_path)
for edge in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [7, "red"])
    edge = Topology.SetDictionary(edge, d)
edges = Topology.Edges(straight_path)
for edge in edges:
    d = Dictionary.ByKeysValues(["width", "color"], [7, "blue"])
    edge = Topology.SetDictionary(edge, d)

In [ ]:
Topology.Show(police_station, shortest_path, straight_path,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColorKey="color",
              edgeWidthKey="width",
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

### //13c Closeness Centrality | Integration
> Measures topological shallowness — how close each space is to all others. In space syntax: global integration.

In [ ]:
centrality_list = Graph.ClosenessCentrality(analysis_graph, colorScale="thermal")

> Transfer results from graph vertices back to shell faces.

In [ ]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [ ]:
Topology.Show(faces,
              faceColorKey="cc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

### //13d Betweenness Centrality | Choice
> Measures how often a node lies on shortest paths between others. In space syntax: choice.

In [ ]:
centrality_list = Graph.BetweennessCentrality(analysis_graph, normalize=True, colorScale="thermal")

> Transfer results from graph vertices back to shell faces.

In [ ]:
reset_dictionaries(shell)
faces = Topology.Faces(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [ ]:
Topology.Show(faces,
              faceColorKey="bc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)